# ChargebackOps Merchant Agent - outcome-based RL on Qwen2.5-3B (fp16 LoRA)

End-to-end pipeline for a single Colab T4 (or Kaggle T4):

1. **Phase A - JSON SFT** on heuristic rollouts. Teaches the model the env action schema.
2. **Phase B - GRPO with outcome reward**. Reward = terminal `$` PnL after the model's action plus heuristic tail-rollout. The merchant policy is pushed toward actions that *win money* against the scripted Issuer + arbitration, not actions that match the heuristic. This is real RLVR (verifiable rewards) - the verifier is the dispute outcome.
3. **Eval** every checkpoint against the heuristic baseline + naive baseline.

**Why outcome reward over heuristic-match**: heuristic-match training is supervised distillation in disguise - the model can never beat the teacher and the reward is gameable by mimicry. Outcome reward is dollar-denominated, adversarially-verified by the Issuer, and ungameable: the only way to earn it is to actually win disputes.

**Theme alignment**: Multi-Agent (merchant vs scripted Issuer) primary; Professional Tasks (B2B chargeback workflow) and Long-Horizon (multi-round arbitration) secondary.

Model: `Qwen/Qwen2.5-3B-Instruct` fp16 + LoRA r=16. Fits T4 with no bitsandbytes.

## 0. Setup - install deps + clone repo

In [ ]:
# GPU + repo setup for Colab T4 (also runs on Kaggle T4).
# Pin the training stack to the combo that supports both:
#   - SFTTrainer.compute_loss without the shape-bug regression
#   - vanilla GRPOTrainer with reward_funcs returning per-completion floats
# Newer Colab/Kaggle images preinstall transformers 5.x + hub 1.x; this cell
# isolates a self-consistent training stack in a private deps directory.
import os
import shutil
import subprocess
import sys
import importlib

if os.path.isdir('/content'):
    WORK_DIR = '/content'
elif os.path.isdir('/kaggle/working'):
    WORK_DIR = '/kaggle/working'
else:
    raise RuntimeError('Notebook expects Colab or Kaggle. Set WORK_DIR manually otherwise.')
os.chdir(WORK_DIR)
DRIVE_ROOT = '/content/drive/MyDrive'
PERSIST_ROOT = (
    os.path.join(DRIVE_ROOT, 'chargebackops-artifacts')
    if os.path.isdir(DRIVE_ROOT) else WORK_DIR
)
os.makedirs(PERSIST_ROOT, exist_ok=True)
print('work dir:', WORK_DIR)
print('artifact root:', PERSIST_ROOT)
print(subprocess.check_output(['nvidia-smi', '-L']).decode())

PYTHON = sys.executable
PIP = [PYTHON, '-m', 'pip']
DEPS_DIR = os.path.join(WORK_DIR, '_pydeps')
if os.path.isdir(DEPS_DIR):
    shutil.rmtree(DEPS_DIR)
os.makedirs(DEPS_DIR, exist_ok=True)
print('python:', PYTHON)
print('pip cmd:', ' '.join(PIP))
print('deps dir:', DEPS_DIR)

# STEP 1 - torch trio for cu128. Turing T4 supports cu118+; cu128 wheels work.
subprocess.run(
    PIP + ['install', '-q', '--no-cache-dir',
           'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
           '--index-url', 'https://download.pytorch.org/whl/cu128'],
    check=True, cwd=WORK_DIR,
)

CORE_STACK = [
    'transformers==4.51.3',
    'trl==0.21.0',
    'peft==0.14.0',
    'accelerate==1.0.1',
    'tokenizers==0.21.4',
    'huggingface-hub==0.30.2',
]

# STEP 2a - install the exact Hugging Face training stack into a private deps
# directory. This avoids Colab/Kaggle system packages shadowing pinned wheels.
# transformers 4.51.3 requires huggingface-hub>=0.30,<1.0, so 0.30.2 is a
# consistent lower-end hub pin for this stack.
subprocess.run(
    PIP + ['install', '-q', '--no-cache-dir', '--upgrade', '--target', DEPS_DIR, '--no-deps'] + CORE_STACK,
    check=True, cwd=WORK_DIR,
)

# STEP 2b - supporting libs into the same private deps dir. This can still drag
# newer transitive HF packages into DEPS_DIR, so the core stack is cleaned and
# re-applied immediately afterward.
subprocess.run(
    PIP + ['install', '-q', '--upgrade', '--target', DEPS_DIR, '--upgrade-strategy=only-if-needed',
           'datasets>=2.20,<4.0',
           'matplotlib>=3.8',
           'pydantic>=2.10',
           'openenv-core>=0.2.2'],
    check=True, cwd=WORK_DIR,
)

# STEP 2c - remove any shadowing core-package copies added by supporting deps,
# then reinstall the exact training stack into DEPS_DIR.
CORE_TOPLEVEL = {'transformers', 'tokenizers', 'trl', 'peft', 'accelerate', 'huggingface_hub'}
for name in list(os.listdir(DEPS_DIR)):
    stem = name.split('-')[0]
    if stem in CORE_TOPLEVEL:
        path = os.path.join(DEPS_DIR, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
subprocess.run(
    PIP + ['install', '-q', '--no-cache-dir', '--upgrade', '--target', DEPS_DIR, '--no-deps'] + CORE_STACK,
    check=True, cwd=WORK_DIR,
)

# Make the private deps directory shadow system site-packages for this kernel
# and for any child Python processes started later.
os.environ['PYTHONPATH'] = DEPS_DIR + os.pathsep + os.environ.get('PYTHONPATH', '')
if DEPS_DIR not in sys.path:
    sys.path.insert(0, DEPS_DIR)
importlib.invalidate_caches()
for mod in list(sys.modules):
    if mod.split('.')[0] in {'huggingface_hub', 'transformers', 'tokenizers', 'trl', 'peft', 'accelerate', 'datasets'}:
        del sys.modules[mod]

# Clone repo (always fresh) so the editable install matches main.
REPO_DIR = os.path.join(WORK_DIR, 'chargebackops')
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ['git', 'clone', '--depth', '1',
     'https://github.com/MitudruDutta/ChargeBackOps.git', REPO_DIR],
    check=True, cwd=WORK_DIR,
)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

# Editable install with --no-deps so pyproject's app/server deps do not alter
# the training stack we just pinned.
subprocess.run(PIP + ['install', '-q', '-e', '.', '--no-deps'],
               check=True, cwd=REPO_DIR)

# Verify the runtime imports are coming from the private deps directory.
import importlib.metadata as md_
PINNED_DIST_NAMES = {'transformers', 'tokenizers', 'trl', 'peft', 'accelerate', 'huggingface-hub'}
_original_md_version = md_.version

def _deps_dir_version(pkg_name):
    normalized = pkg_name.lower().replace('_', '-')
    if normalized in PINNED_DIST_NAMES:
        for dist in md_.distributions(path=[DEPS_DIR]):
            if (dist.metadata.get('Name') or '').lower() == normalized:
                return dist.version
    return _original_md_version(pkg_name)

md_.version = _deps_dir_version
import importlib.metadata
importlib.metadata.version = _deps_dir_version

import huggingface_hub
import transformers
import tokenizers
import trl
import peft
import accelerate

print('torch           ', md_.version('torch'))
print('torchvision     ', md_.version('torchvision'))
print('transformers    ', transformers.__version__, transformers.__file__)
print('tokenizers      ', tokenizers.__version__, tokenizers.__file__)
print('huggingface_hub ', huggingface_hub.__version__, huggingface_hub.__file__)
print('trl             ', trl.__version__, trl.__file__)
print('peft            ', peft.__version__, peft.__file__)
print('accelerate      ', accelerate.__version__, accelerate.__file__)
print('openenv-core    ', md_.version('openenv-core'))
assert transformers.__version__ == '4.51.3', 'transformers pin failed'
assert tokenizers.__version__.startswith('0.21'), 'tokenizers pin failed'
assert huggingface_hub.__version__ == '0.30.2', 'huggingface_hub runtime pin failed'
assert trl.__version__ == '0.21.0', 'trl pin failed'
assert peft.__version__ == '0.14.0', 'peft pin failed'
assert accelerate.__version__ == '1.0.1', 'accelerate pin failed'
assert os.path.realpath(DEPS_DIR) in os.path.realpath(huggingface_hub.__file__), 'huggingface_hub not loaded from DEPS_DIR'


work dir: /content
artifact root: /content
GPU 0: Tesla T4 (UUID: GPU-cbe1e490-2f3c-56f5-7aa1-d4d7f23ce79a)

python: /usr/bin/python3
pip cmd: /usr/bin/python3 -m pip
deps dir: /content/_pydeps
cwd: /content/chargebackops
torch            2.10.0+cu128
torchvision      0.25.0+cu128
transformers     4.51.3 /content/_pydeps/transformers/__init__.py
tokenizers       0.21.4 /content/_pydeps/tokenizers/__init__.py
huggingface_hub  0.30.2 /content/_pydeps/huggingface_hub/__init__.py
trl              0.21.0 /content/_pydeps/trl/__init__.py
peft             0.14.0 /content/_pydeps/peft/__init__.py
accelerate       1.0.1 /content/_pydeps/accelerate/__init__.py
openenv-core     0.2.3


In [ ]:
# Path + module-cache flush so the editable install resolves before any other import.
import os, sys, importlib, logging, torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Prefer the notebook-local pinned deps dir over any Colab/Kaggle system wheels.
DEPS_DIR = globals().get('DEPS_DIR') or os.path.join('/content', '_pydeps')
if os.path.isdir(DEPS_DIR) and DEPS_DIR not in sys.path:
    sys.path.insert(0, DEPS_DIR)

# Silence transformers per-layer "Caching is incompatible..." spam at scale
# (every GRPO rollout fires it once per layer otherwise, hiding loss logs).
import transformers
transformers.logging.set_verbosity_error()
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('transformers.models.qwen2.modeling_qwen2').setLevel(logging.ERROR)

REPO_DIR = globals().get('REPO_DIR') or (
    '/content/chargebackops' if os.path.isdir('/content/chargebackops')
    else '/kaggle/working/chargebackops'
)
PERSIST_ROOT = globals().get('PERSIST_ROOT') or (
    os.path.join('/content/drive/MyDrive', 'chargebackops-artifacts')
    if os.path.isdir('/content/drive/MyDrive') else os.path.dirname(REPO_DIR)
)
os.makedirs(PERSIST_ROOT, exist_ok=True)
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Repository checkout missing at {REPO_DIR}. Run setup first.')
sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for mod in list(sys.modules):
    if mod.startswith(('scenarios', 'training', 'evaluation', 'server', 'core', 'runners', 'connectors')):
        del sys.modules[mod]
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('repo:', REPO_DIR)
print('deps dir:', DEPS_DIR)
print('artifact root:', PERSIST_ROOT)

torch 2.10.0+cu128 | cuda True Tesla T4
repo: /content/chargebackops
deps dir: /content/_pydeps
artifact root: /content


## 1. Load Qwen2.5-3B-Instruct fp16 + attach LoRA

* `dtype=torch.float16` - T4 = Turing sm_75, no bf16 hardware.
* LoRA r=16 on q/k/v/o + gate/up/down. ~9M trainable params.
* `gradient_checkpointing` + `enable_input_require_grads` keeps activations off VRAM during backward.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = os.environ.get('MODEL_ID', 'Qwen/Qwen2.5-3B-Instruct')
print('MODEL_ID:', MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # required for GRPO generation

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

lora_target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj']
lora_rank = 16
lora_alpha = 32

lora_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print(f'VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | '
      f'free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

MODEL_ID: Qwen/Qwen2.5-3B-Instruct


/content/_pydeps/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
VRAM allocated: 6.29 GB | free: 8.50 GB


## 2. Phase A - SFT on heuristic rollouts

Builds (prompt, oracle_completion) pairs by rolling the scripted heuristic on every headline + generated task. Wraps in the Qwen chat template so the model learns the same prompt format used at inference. After this phase the model emits valid JSON with the right `action_type` per state - solves the *always emit `select_case`* collapse before GRPO ever runs.

In [ ]:
from datasets import Dataset
from scenarios.simulation import list_tasks, get_task
from training.sft_dataset import build_sft_dataset
from collections import Counter

# Synthetic pool. Default 10k rows for T4 speed; override via env var.
SFT_TARGET_ROWS = int(os.environ.get('SFT_TARGET_ROWS', '4000'))
SFT_MAX_ROWS = int(os.environ.get('SFT_MAX_ROWS', str(SFT_TARGET_ROWS)))
SFT_SEED_START = int(os.environ.get('SFT_SEED_START', '1000'))
SFT_SEED_BATCH = int(os.environ.get('SFT_SEED_BATCH', '128'))
SFT_MAX_STATES_PER_TASK = int(os.environ.get('SFT_MAX_STATES_PER_TASK', '24'))
GRPO_SEED_COUNT = int(os.environ.get('GRPO_SEED_COUNT', '160'))

# Holdout seeds excluded from training so eval is defensible.
HOLDOUT_SEEDS_BY_DIFF = {
    'easy': {42},
    'medium': {17, 99},
    'hard': {7, 53},
    'nightmare': {31, 77},
}
DIFFICULTIES = ['easy', 'medium', 'hard', 'nightmare']

headline_task_ids = [t.task_id for t in list_tasks()]
task_ids = list(headline_task_ids)
raw_sft = build_sft_dataset(headline_task_ids, max_states_per_task=SFT_MAX_STATES_PER_TASK)
generated_train_task_ids = []

seed_cursor = SFT_SEED_START
while len(raw_sft) < SFT_TARGET_ROWS:
    batch_task_ids = []
    for diff in DIFFICULTIES:
        blocked = HOLDOUT_SEEDS_BY_DIFF.get(diff, set())
        for seed in range(seed_cursor, seed_cursor + SFT_SEED_BATCH):
            if seed in blocked:
                continue
            tid = f'generated_{diff}_s{seed}'
            get_task(tid)
            batch_task_ids.append(tid)
    raw_sft.extend(build_sft_dataset(batch_task_ids, max_states_per_task=SFT_MAX_STATES_PER_TASK))
    generated_train_task_ids.extend(batch_task_ids)
    task_ids.extend(batch_task_ids)
    seed_cursor += SFT_SEED_BATCH
    print(f'generated SFT rows: {len(raw_sft):,} / target {SFT_TARGET_ROWS:,}')

if len(raw_sft) > SFT_MAX_ROWS:
    raw_sft = raw_sft[:SFT_MAX_ROWS]

# Seed list for GRPO state-action curriculum (smaller than SFT pool because
# GRPO rollouts are slower than SFT forward passes).
seeds = list(range(SFT_SEED_START, SFT_SEED_START + GRPO_SEED_COUNT))

def to_chat_text(prompt, completion):
    return tokenizer.apply_chat_template(
        [
            {'role': 'user', 'content': prompt},
            {'role': 'assistant', 'content': completion},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

sft_rows = [{'text': to_chat_text(s['prompt'], s['completion'])} for s in raw_sft]
sft_dataset = Dataset.from_list(sft_rows)

atype_counts = Counter(s['action_type'] for s in raw_sft)
print(f'SFT samples: {len(sft_dataset):,}, unique tasks: {len(set(s["task_id"] for s in raw_sft)):,}')
print(f'headline tasks: {len(headline_task_ids)}, generated train tasks used: {len(generated_train_task_ids):,}')
print(f'excluded generated holdout seeds: {HOLDOUT_SEEDS_BY_DIFF}')
print(f'action_type distribution: {dict(atype_counts)}')
print('sample (first 500 chars):')
print(sft_rows[0]['text'][:500])

generated SFT rows: 6,091 / target 4,000
SFT samples: 4,000, unique tasks: 414
headline tasks: 12, generated train tasks used: 512
excluded generated holdout seeds: {'easy': {42}, 'medium': {17, 99}, 'hard': {53, 7}, 'nightmare': {77, 31}}
action_type distribution: {'select_case': 828, 'query_system': 972, 'add_evidence': 377, 'set_strategy': 620, 'submit_representment': 375, 'retrieve_policy': 244, 'respond_to_pre_arb': 139, 'resolve_case': 445}
sample (first 500 chars):
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
You play the merchant-side agent in a chargeback dispute. Look at the observation and choose the single best next action. Return JSON only: {"action_type": "...", "case_id": "...", "strategy": "...", "evidence_ids": [...], "note": "..."} Use only action_types listed in available_actions. Omit fields you do not need.
OBSERVATION:
{"available_actions":["select_case"],"last_action_resu


In [ ]:
from trl import SFTConfig, SFTTrainer

OUT_ROOT = PERSIST_ROOT
SFT_DIR = os.path.join(OUT_ROOT, 'sft-merchant-agent')
GRPO_DIR = os.path.join(OUT_ROOT, 'grpo-merchant-agent')

SFT_FINAL_DIR = os.path.join(SFT_DIR, 'final')
RUN_SFT_TRAIN = os.environ.get('RUN_SFT_TRAIN', 'auto').strip().lower()
TRAIN_SFT = RUN_SFT_TRAIN in {'1', 'true', 'yes', 'y', 'on'} or (
    RUN_SFT_TRAIN == 'auto' and not os.path.isdir(SFT_FINAL_DIR)
)
SFT_EPOCHS = float(os.environ.get('SFT_EPOCHS', '1'))
SFT_LR = float(os.environ.get('SFT_LR', '1e-4'))
SFT_MAX_STEPS = int(os.environ.get('SFT_MAX_STEPS', '300'))

if not TRAIN_SFT:
    print(f'Skipping SFT train; using existing adapter at {SFT_FINAL_DIR}')
else:
    sft_config = SFTConfig(
        output_dir=SFT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=SFT_EPOCHS,
        max_steps=SFT_MAX_STEPS,
        learning_rate=SFT_LR,
        logging_steps=10,
        save_steps=300,
        save_total_limit=2,
        bf16=False,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        max_length=1024,
        dataset_text_field='text',
        report_to='none',
        optim='adamw_torch',
        warmup_ratio=0.03,
    )
    print(f'SFT config: rows={len(sft_dataset):,}, epochs={SFT_EPOCHS}, lr={SFT_LR}, max_steps={SFT_MAX_STEPS}')
    if hasattr(model, 'config'):
        model.config.use_cache = False
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )
    sft_trainer.train()
    sft_trainer.save_model(SFT_FINAL_DIR)
    del sft_trainer
    torch.cuda.empty_cache()
    print(f'PEAK VRAM (SFT): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

SFT config: rows=4,000, epochs=1.0, lr=0.0001, max_steps=300


Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

{'loss': 2.7943, 'grad_norm': 1.1976054906845093, 'learning_rate': 0.0001, 'num_tokens': 33488.0, 'mean_token_accuracy': 0.5481752466410399, 'epoch': 0.02}
{'loss': 1.4606, 'grad_norm': 1.046560525894165, 'learning_rate': 9.656357388316152e-05, 'num_tokens': 69410.0, 'mean_token_accuracy': 0.6945387788116932, 'epoch': 0.04}
{'loss': 0.6129, 'grad_norm': 0.6730246543884277, 'learning_rate': 9.312714776632303e-05, 'num_tokens': 104689.0, 'mean_token_accuracy': 0.8604250960052013, 'epoch': 0.06}
{'loss': 0.3365, 'grad_norm': 0.6759085059165955, 'learning_rate': 8.969072164948454e-05, 'num_tokens': 138660.0, 'mean_token_accuracy': 0.9138240091502666, 'epoch': 0.08}
{'loss': 0.2353, 'grad_norm': 0.4965641498565674, 'learning_rate': 8.625429553264606e-05, 'num_tokens': 172382.0, 'mean_token_accuracy': 0.9320120006799698, 'epoch': 0.1}
{'loss': 0.1764, 'grad_norm': 0.40537405014038086, 'learning_rate': 8.281786941580757e-05, 'num_tokens': 207064.0, 'mean_token_accuracy': 0.9419497564435005, '

## 2.5. Merge SFT LoRA into base, attach fresh LoRA for GRPO

`accelerate.unwrap_model_for_generation` calls `merge_adapter()` + `unmerge_adapter()` around generation. With fp16 LoRA the round-trip can lose enough precision that completions degrade. Fix: bake SFT into base via `merge_and_unload()`, then attach a fresh zero-initialized LoRA. The fresh adapter starts as identity, so generation emits SFT-quality output regardless of TRL adapter toggling. GRPO then trains the fresh adapter on top.

In [ ]:
# Reload saved SFT LoRA into a fresh base, merge it, then attach a fresh Phase B LoRA.
from peft import PeftModel
import gc

if not os.path.isdir(SFT_FINAL_DIR):
    raise FileNotFoundError(f'Missing SFT adapter: {SFT_FINAL_DIR}. Run Phase A or upload the adapter first.')

for name in ['model', 'base_model', 'merged_base']:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
print(f'before SFT reload: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

fresh_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
sft_model = PeftModel.from_pretrained(fresh_base, SFT_FINAL_DIR)
merged_base = sft_model.merge_and_unload()
del sft_model, fresh_base
gc.collect()
torch.cuda.empty_cache()
print(f'after SFT merge: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')
merged_base.enable_input_require_grads()

# Sanity: SFT-baked base should emit clean JSON deterministically.
from training.env_adapter import build_prompt
from server.chargeback_ops_environment import ChargebackOpsEnvironment
env = ChargebackOpsEnvironment()
obs = env.reset(task_id='goods_not_received_easy')
chat = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': build_prompt(obs.model_dump())}],
    tokenize=False, add_generation_prompt=True,
)
inp = tokenizer(chat, return_tensors='pt').to(merged_base.device)
merged_base.eval()
with torch.no_grad():
    out = merged_base.generate(
        **inp,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print('merged-base gen:', repr(tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=False)))
merged_base.train()

# Attach fresh Phase B LoRA. lora_dropout=0.1 keeps stochasticity ALIVE during
# train()-mode generation. The v1 attempt set this to 0 + low temp + small
# num_generations - result was every group of 4 emitted identical completions,
# std=0, GRPO advantage=0, no learning. With dropout=0.1 plus temp=1.3 +
# num_generations=8 set in the GRPO cell, within-group variance is restored.
lora_phase_b = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(merged_base, lora_phase_b)
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f'after fresh Phase B LoRA: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')


before SFT reload: VRAM 0.02 GB


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

after SFT merge: VRAM 6.19 GB


/content/_pydeps/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/content/_pydeps/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/content/_pydeps/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


merged-base gen: '{"action_type":"select_case","case_id":"CB-E1","metadata":{}}<|im_end|>'
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
after fresh Phase B LoRA: VRAM 6.31 GB


## 3. Phase B - GRPO with outcome reward (RLVR, not distillation)

Reward source: terminal `$` PnL after the model's action plus heuristic tail-rollout. The merchant earns positive reward only when its action leads to a winning packet against the scripted Issuer or arbitration. A second format-shaping reward provides dense early-training signal so GRPO has gradient before the policy can produce winning packets.

It optimizes "win the dispute." The Issuer + arbitration are the verifier - they cannot be tricked because reward is dollar-denominated outcome.

In [ ]:
from training.reward_adapter import build_state_action_dataset

# State-action samples: (task_id, state_step, prompt) tuples captured by
# rolling the heuristic forward on each task. The model will be asked to
# pick the next action at each captured state.
PHASE_B_MAX_STATES_PER_TASK = int(os.environ.get('PHASE_B_MAX_STATES_PER_TASK', '10'))
GRPO_DIFFICULTIES = tuple(
    d.strip()
    for d in os.environ.get('GRPO_DIFFICULTIES', 'easy,medium,hard,nightmare').split(',')
    if d.strip()
)
curriculum_task_ids = [
    t.task_id for t in list_tasks()
    if t.difficulty in GRPO_DIFFICULTIES
]
for diff in GRPO_DIFFICULTIES:
    blocked = HOLDOUT_SEEDS_BY_DIFF.get(diff, set())
    for s in seeds:
        if s in blocked:
            continue
        tid = f'generated_{diff}_s{s}'
        try:
            get_task(tid)
            if tid not in curriculum_task_ids:
                curriculum_task_ids.append(tid)
        except Exception:
            pass

raw_grpo = build_state_action_dataset(
    curriculum_task_ids, max_states_per_task=PHASE_B_MAX_STATES_PER_TASK,
)

def to_chat_prompt(prompt):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False, add_generation_prompt=True,
    )

grpo_rows = []
for sample in raw_grpo:
    chat_prompt = to_chat_prompt(sample['prompt'])
    n_tokens = len(tokenizer(chat_prompt, add_special_tokens=False)['input_ids'])
    if n_tokens <= 1000:
        grpo_rows.append({
            'prompt': chat_prompt,
            'task_id': sample['task_id'],
            'state_step': int(sample['state_step']),
        })

grpo_dataset = Dataset.from_list(grpo_rows)
unique_tasks = len({row['task_id'] for row in grpo_rows})
print(f'GRPO state-action samples: {len(grpo_dataset)}, '
      f'unique tasks: {unique_tasks}, difficulties={GRPO_DIFFICULTIES}, '
      f'max_states_per_task={PHASE_B_MAX_STATES_PER_TASK}')

GRPO state-action samples: 5638, unique tasks: 652, difficulties=('easy', 'medium', 'hard', 'nightmare'), max_states_per_task=10


In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.outcome_reward import compute_outcome_reward, compute_format_reward

# Re-arm hooks and disable cache. Do NOT zero out dropout - the merge cell
# attached the Phase B LoRA with lora_dropout=0.1 specifically so train()-mode
# generation has stochasticity. Zeroing it here was the v1 bug.
model.enable_input_require_grads()
if hasattr(model, 'config'):
    model.config.use_cache = False
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model, 'generation_config'):
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id

def outcome_reward_fn(prompts, completions, **kwargs):
    task_ids = kwargs.get('task_id') or kwargs.get('task_ids')
    state_steps = kwargs.get('state_step') or kwargs.get('state_steps')
    return compute_outcome_reward(
        prompts, completions,
        task_ids=task_ids, state_steps=state_steps,
    )

def format_reward_fn(prompts, completions, **kwargs):
    return compute_format_reward(prompts, completions)

# CRITICAL sampling kwargs - rewritten after the v1 run had grad_norm=0.0 on
# 95% of steps. The v1 logs showed:
#   - frac_reward_zero_std=1.0 on ~80% of steps (all 4 generations identical)
#   - entropy=0.001-0.017 (policy near-delta after SFT mean_acc=0.96)
#   - When std=0 inside a group, advantage=0 and gradient=0.
#
# Fix: aggressively widen the sampling distribution.
#   temperature: 0.7 -> 1.3   (past 1.0 breaks the SFT argmax lock)
#   top_p:       0.9 -> 1.0   (no nucleus truncation)
#   top_k:       50  -> 0     (no top-k truncation)
#   num_generations: 4 -> 8   (2x within-group variance odds; gen_batch=8 must be divisible)
#   learning_rate: 5e-6 -> 2e-5 (bigger push to escape SFT collapse)
#   beta:        0.0 -> 0.04  (small KL anchor; v1 collapse risk is gone)
#   lora_dropout: 0.0 -> 0.1  (set in the merge cell, kept here)
#
# The format_reward_fn (-0.10 for invalid JSON) is the safety net that stops
# the model from drifting into pure noise at the higher temperature.
grpo_config = GRPOConfig(
    output_dir=GRPO_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=8,
    max_prompt_length=1024,
    max_completion_length=192,
    learning_rate=float(os.environ.get('GRPO_LR', '2e-5')),
    max_steps=int(os.environ.get('GRPO_MAX_STEPS', '60')),
    logging_steps=5,
    save_steps=60,
    save_total_limit=2,
    bf16=False,
    fp16=True,
    max_grad_norm=0.5,
    gradient_checkpointing=False,
    report_to='none',
    beta=0.04,
    temperature=1.3,
    top_p=1.0,
    top_k=0,
    repetition_penalty=1.0,
    use_vllm=False,
    log_completions=True,
    num_completions_to_print=2,
    optim='adamw_torch',
    lr_scheduler_type='constant',
)

RUN_GRPO = os.environ.get('RUN_GRPO', '1').strip().lower() not in {'0', 'false', 'no'}
if RUN_GRPO:
    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[outcome_reward_fn, format_reward_fn],
        args=grpo_config,
        train_dataset=grpo_dataset,
    )
    grpo_trainer.train()
    grpo_trainer.save_model(os.path.join(GRPO_DIR, 'final'))
    del grpo_trainer
    torch.cuda.empty_cache()
else:
    print('RUN_GRPO=0: skipped GRPO training')
print(f'PEAK VRAM (GRPO): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')


{'loss': 0.0, 'grad_norm': 3.04646641779982e-06, 'learning_rate': 2e-05, 'num_tokens': 19323.0, 'completions/mean_length': 25.075, 'completions/min_length': 22.8, 'completions/max_length': 25.4, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 25.075, 'completions/min_terminated_length': 22.8, 'completions/max_terminated_length': 25.4, 'rewards/outcome_reward_fn/mean': -0.39629372358322146, 'rewards/outcome_reward_fn/std': 0.17307148277759551, 'rewards/format_reward_fn/mean': 0.05000000074505806, 'rewards/format_reward_fn/std': 0.0, 'reward': -0.3462937235832214, 'reward_std': 0.1730714797973633, 'frac_reward_zero_std': 0.6, 'kl': 7.951766159530749e-07, 'entropy': 0.03927592048421502, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'epoch': 0.0008868393047179851}


╭──────────────────────────────────────────────────── Step 5 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                      ┃ Completion                 ┃ outcome_reward_fn ┃ format_reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ <|im_start|>system          │ {"action_type":"query_sys… │             -0.40 │             0.05 │      0.00 │ │
│ │ You are Qwen, created by    │                            │                   │                  │           │ │
│ │ Alibaba Cloud. You are a    │                            │                   │                  │           │ │
│ │ helpful                     │                            │                   │                  │           │ │
│ │ assistant.<|im_end|>        │                            │                   │                  │           │ │
│ │ <|im_start|>user            │                            │                   │                  │           │ │
│ │ You play the merchant-side  │                            │                   │                  │           │ │
│ │ agent in a chargeback       │                            │                   │                  │           │ │
│ │ dispute. Look at the        │                            │                   │                  │           │ │
│ │ observation and choose the  │                            │                   │                  │           │ │
│ │ single best next action.    │                            │                   │                  │           │ │
│ │ Return JSON only:           │                            │                   │                  │           │ │
│ │ {"action_type": "...",      │                            │                   │                  │           │ │
│ │ "case_id": "...",           │                            │                   │                  │           │ │
│ │ "strategy": "...",          │                            │                   │                  │           │ │
│ │ "evidence_ids": [...],      │                            │                   │                  │           │ │
│ │ "note": "..."} Use only     │                            │                   │                  │           │ │
│ │ action_types listed in      │                            │                   │                  │           │ │
│ │ available_actions. Omit     │                            │                   │                  │           │ │
│ │ fields you do not need.     │                            │                   │                  │           │ │
│ │ OBSERVATION:                │                            │                   │                  │           │ │
│ │ {"available_actions":["sel… │                            │                   │                  │           │ │
│ │ orders for case CB-G3;      │                            │                   │                  │           │ │
│ │ found 1 evidence items,     │                            │                   │                  │           │ │
│ │ including 1 useful          │                            │                   │                  │           │ │
│ │ ones.","objective":"Survive │                            │                   │                  │           │ │
│ │ 5 disputes                  │                            │                   │                  │           │ │
│ │ (duplicate_processing,      │                            │                   │                  │           │ │
│ │ fraud_cnp,                  │                            │                   │                  │           │ │
│ │ goods_not_received) with    │                            │                   │                  │           │ │
│ │ adversarial evidence,       │                       

{'loss': 0.0, 'grad_norm': 0.0004595154314301908, 'learning_rate': 2e-05, 'num_tokens': 35724.0, 'completions/mean_length': 32.025, 'completions/min_length': 31.2, 'completions/max_length': 35.4, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 32.025, 'completions/min_terminated_length': 31.2, 'completions/max_terminated_length': 35.4, 'rewards/outcome_reward_fn/mean': 0.3736898750066757, 'rewards/outcome_reward_fn/std': 0.07071067690849304, 'rewards/format_reward_fn/mean': 0.04625000059604645, 'rewards/format_reward_fn/std': 0.01060660183429718, 'reward': 0.41993985772132875, 'reward_std': 0.08131728172302247, 'frac_reward_zero_std': 0.8, 'kl': 0.00024051719977666864, 'entropy': 0.04845021460205316, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'epoch': 0.0017736786094359701}


╭──────────────────────────────────────────────────── Step 10 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                     ┃ Completion                  ┃ outcome_reward_fn ┃ format_reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ <|im_start|>system         │ {"action_type":"set_strate… │             -0.91 │             0.05 │      0.00 │ │
│ │ You are Qwen, created by   │                             │                   │                  │           │ │
│ │ Alibaba Cloud. You are a   │                             │                   │                  │           │ │
│ │ helpful                    │                             │                   │                  │           │ │
│ │ assistant.<|im_end|>       │                             │                   │                  │           │ │
│ │ <|im_start|>user           │                             │                   │                  │           │ │
│ │ You play the merchant-side │                             │                   │                  │           │ │
│ │ agent in a chargeback      │                             │                   │                  │           │ │
│ │ dispute. Look at the       │                             │                   │                  │           │ │
│ │ observation and choose the │                             │                   │                  │           │ │
│ │ single best next action.   │                             │                   │                  │           │ │
│ │ Return JSON only:          │                             │                   │                  │           │ │
│ │ {"action_type": "...",     │                             │                   │                  │           │ │
│ │ "case_id": "...",          │                             │                   │                  │           │ │
│ │ "strategy": "...",         │                             │                   │                  │           │ │
│ │ "evidence_ids": [...],     │                             │                   │                  │           │ │
│ │ "note": "..."} Use only    │                             │                   │                  │           │ │
│ │ action_types listed in     │                             │                   │                  │           │ │
│ │ available_actions. Omit    │                             │                   │                  │           │ │
│ │ fields you do not need.    │                             │                   │                  │           │ │
│ │ OBSERVATION:               │                             │                   │                  │           │ │
│ │ {"available_actions":["se… │                             │                   │                  │           │ │
│ │ case                       │                             │                   │                  │           │ │
│ │ CB-G2.","objective":"Opti… │                             │                   │                  │           │ │
│ │ outcomes across 3 disputes │                             │                   │                  │           │ │
│ │ (goods_not_received,       │                             │                   │                  │           │ │
│ │ duplicate_processing,      │                             │                   │                  │           │ │
│ │ fraud_cnp) under tight     │                             │                   │                  │           │ │
│ │ deadlines. Prioritize      │                             │                   │                  │           │ │
│ │ high-value recoverable     │                             │                   │                  │           │ │
│ │ cases and concede weak     │                        

{'loss': 0.0005, 'grad_norm': 3.3480523597972933e-07, 'learning_rate': 2e-05, 'num_tokens': 52387.0, 'completions/mean_length': 30.975, 'completions/min_length': 22.4, 'completions/max_length': 36.8, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 30.975, 'completions/min_terminated_length': 22.4, 'completions/max_terminated_length': 36.8, 'rewards/outcome_reward_fn/mean': 0.17375480830669404, 'rewards/outcome_reward_fn/std': 0.22531243227422237, 'rewards/format_reward_fn/mean': 0.04625000059604645, 'rewards/format_reward_fn/std': 0.01060660183429718, 'reward': 0.22000478506088256, 'reward_std': 0.2359190309420228, 'frac_reward_zero_std': 0.2, 'kl': 0.013398571864327014, 'entropy': 0.11388611167203636, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'epoch': 0.0026605179141539555}


...

{'loss': 0.0011, 'grad_norm': 2.68776798248291, 'learning_rate': 2e-05, 'num_tokens': 211499.0, 'completions/mean_length': 26.25, 'completions/min_length': 20.6, 'completions/max_length': 48.0, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 26.25, 'completions/min_terminated_length': 20.6, 'completions/max_terminated_length': 48.0, 'rewards/outcome_reward_fn/mean': 0.10574271269142628, 'rewards/outcome_reward_fn/std': 0.21722247805446387, 'rewards/format_reward_fn/mean': 0.03500000052154064, 'rewards/format_reward_fn/std': 0.02613307684659958, 'reward': 0.14074270175769926, 'reward_std': 0.2229499489068985, 'frac_reward_zero_std': 0.4, 'kl': 0.027717302633544348, 'entropy': 0.06688877020496875, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'epoch': 0.010642071656615822}


╭──────────────────────────────────────────────────── Step 60 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                      ┃ Completion                 ┃ outcome_reward_fn ┃ format_reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ <|im_start|>system          │ {"action_type":"set_strat… │              1.00 │             0.05 │      0.72 │ │
│ │ You are Qwen, created by    │                            │                   │                  │           │ │
│ │ Alibaba Cloud. You are a    │                            │                   │                  │           │ │
│ │ helpful                     │                            │                   │                  │           │ │
│ │ assistant.<|im_end|>        │                            │                   │                  │           │ │
│ │ <|im_start|>user            │                            │                   │                  │           │ │
│ │ You play the merchant-side  │                            │                   │                  │           │ │
│ │ agent in a chargeback       │                            │                   │                  │           │ │
│ │ dispute. Look at the        │                            │                   │                  │           │ │
│ │ observation and choose the  │                            │                   │                  │           │ │
│ │ single best next action.    │                            │                   │                  │           │ │
│ │ Return JSON only:           │                            │                   │                  │           │ │
│ │ {"action_type": "...",      │                            │                   │                  │           │ │
│ │ "case_id": "...",           │                            │                   │                  │           │ │
│ │ "strategy": "...",          │                            │                   │                  │           │ │
│ │ "evidence_ids": [...],      │                            │                   │                  │           │ │
│ │ "note": "..."} Use only     │                            │                   │                  │           │ │
│ │ action_types listed in      │                            │                   │                  │           │ │
│ │ available_actions. Omit     │                            │                   │                  │           │ │
│ │ fields you do not need.     │                            │                   │                  │           │ │
│ │ OBSERVATION:                │                            │                   │                  │           │ │
│ │ {"available_actions":["sel… │                            │                   │                  │           │ │
│ │ evidence G1-BOOKING,        │                            │                   │                  │           │ │
│ │ G1-COMPLETION to case       │                            │                   │                  │           │ │
│ │ CB-G1.","objective":"Resol… │                            │                   │                  │           │ │
│ │ a service_not_provided      │                            │                   │                  │           │ │
│ │ dispute correctly with the  │                            │                   │                  │           │ │
│ │ right evidence before the   │                            │                   │                  │           │ │
│ │ deadline.","queue":[{"amou… │                            │                   │                  │           │ │
│ │ service-not-provided        │                            │                   │                  │           │ │
│ │ disputes when provider      │                       

{'train_runtime': 700.9988, 'train_samples_per_second': 0.685, 'train_steps_per_second': 0.086, 'train_loss': 0.0005090768365927071, 'epoch': 0.010642071656615822}


╭──────────────────────────────────────────────────── Step 60 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                      ┃ Completion                 ┃ outcome_reward_fn ┃ format_reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ <|im_start|>system          │ {"action_type":"submit_re… │             -1.00 │             0.05 │     -1.21 │ │
│ │ You are Qwen, created by    │ completion record and      │                   │                  │           │ │
│ │ Alibaba Cloud. You are a    │ customer acknowledgement   │                   │                  │           │ │
│ │ helpful                     │ confirm the service was    │                   │                  │           │ │
│ │ assistant.<|im_end|>        │ delivered as agreed.       │                   │                  │           │ │
│ │ <|im_start|>user            │ Booking confirmation and   │                   │                  │           │ │
│ │ You play the merchant-side  │ delivery records attached. │                   │                  │           │ │
│ │ agent in a chargeback       │ Supporting evidence:       │                   │                  │           │ │
│ │ dispute. Look at the        │ G1-BOOKING,                │                   │                  │           │ │
│ │ observation and choose the  │ G1-COMPLETION."}           │                   │                  │           │ │
│ │ single best next action.    │                            │                   │                  │           │ │
│ │ Return JSON only:           │                            │                   │                  │           │ │
│ │ {"action_type": "...",      │                            │                   │                  │           │ │
│ │ "case_id": "...",           │                            │                   │                  │           │ │
│ │ "strategy": "...",          │                            │                   │                  │           │ │
│ │ "evidence_ids": [...],      │                            │                   │                  │           │ │
│ │ "note": "..."} Use only     │                            │                   │                  │           │ │
│ │ action_types listed in      │                            │                   │                  │           │ │
│ │ available_actions. Omit     │                            │                   │                  │           │ │
│ │ fields you do not need.     │                            │                   │                  │           │ │
│ │ OBSERVATION:                │                            │                   │                  │           │ │
│ │ {"available_actions":["sel… │                            │                   │                  │           │ │
│ │ evidence G1-BOOKING,        │                            │                   │                  │           │ │
│ │ G1-COMPLETION to case       │                            │                   │                  │           │ │
│ │ CB-G1.","objective":"Resol… │                            │                   │                  │           │ │
│ │ a service_not_provided      │                            │                   │                  │           │ │
│ │ dispute correctly with the  │                            │                   │                  │           │ │
│ │ right evidence before the   │                            │                   │                  │           │ │
│ │ deadline.","queue":[{"amou… │                            │                   │                  │           │ │
│ │ service-not-provided        │                            │                   │                  │           │ │
│ │ disputes when provider      │                       

PEAK VRAM (GRPO): 11.59 GB


## 4. Per-checkpoint eval - overall + per-difficulty

Loads each saved adapter, plays full episodes across the headline catalog, and plots the curve. Stall detection in `run_episode_with_text_policy` ensures degenerate checkpoints reach grading instead of returning 0.

In [ ]:
import gc, os, glob, re
from peft import PeftModel
from training.curve import evaluate_checkpoint, evaluate_checkpoint_by_family

# Free everything (eval_base, sft_merged_base) so GRPO eval has full T4.
for name in ['eval_base', 'sft_merged_base', 'tmp_base', 'sft_for_merge', 'm', 'm_ckpt']:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
print(f'after free: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

OFFLOAD_DIR = '/content/offload'
os.makedirs(OFFLOAD_DIR, exist_ok=True)

# Single fresh base, force everything onto GPU (no auto offload).
fresh = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True,
)
sft_p = PeftModel.from_pretrained(fresh, SFT_FINAL_DIR)
merged = sft_p.merge_and_unload()
del sft_p, fresh
gc.collect(); torch.cuda.empty_cache()
print(f'merged on GPU: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Locate GRPO checkpoint (prefer `final/`, else newest checkpoint-N).
final_grpo = os.path.join(GRPO_DIR, 'final')
grpo_dirs = sorted(
    glob.glob(os.path.join(GRPO_DIR, 'checkpoint-*')),
    key=lambda p: int(re.search(r'checkpoint-(\d+)', p).group(1)),
)
grpo_path = final_grpo if os.path.isdir(final_grpo) else (grpo_dirs[-1] if grpo_dirs else None)
print('grpo path:', grpo_path)

m = PeftModel.from_pretrained(merged, grpo_path, adapter_name='grpo_eval')
m.eval()

eval_tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if eval_tok.pad_token is None:
    eval_tok.pad_token = eval_tok.eos_token
eval_tok.padding_side = 'left'

def grpo_policy(prompt):
    chat = eval_tok.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = eval_tok(chat, return_tensors='pt', truncation=True, max_length=1024).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inputs, max_new_tokens=192, do_sample=False,
            pad_token_id=eval_tok.eos_token_id, eos_token_id=eval_tok.eos_token_id,
        )
    return eval_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

grpo_step = 1 + (max(int(re.search(r'checkpoint-(\d+)', d).group(1)) for d in grpo_dirs) if grpo_dirs else 0) + 1
grpo_overall = evaluate_checkpoint(step=grpo_step, policy=grpo_policy)
grpo_grouped = evaluate_checkpoint_by_family(step=grpo_step, policy=grpo_policy)

print(f'\nGRPO step={grpo_step} mean={grpo_overall.mean_score:.4f}')
print('Per family:')
for fam, ev in sorted(grpo_grouped.by_family.items()):
    print(f'  {fam}={ev.mean_score:.3f}')

# Append to overall/grouped from the previous (partial) eval run.
overall.append(grpo_overall)
grouped.append(grpo_grouped)

print('\nFINAL OVERALL CURVE:')
for c in overall:
    print(f'  step={c.step:4d} mean={c.mean_score:.4f}')

after free: VRAM 1.43 GB


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

merged on GPU: VRAM 7.61 GB
grpo path: /content/grpo-merchant-agent/final

GRPO step=62 mean=0.7283
Per family:
  easy=0.610
  hard=0.817
  medium=0.792
  nightmare=0.694

FINAL OVERALL CURVE:
  step=   0 mean=0.4698
  step=   1 mean=0.7534
  step=  62 mean=0.7283


In [ ]:
from runners.benchmark_runner import run_policy_sweep
sweep = run_policy_sweep()
heur_overall = next(s.mean_score for s in sweep.policies if s.policy == 'heuristic')

from training.curve import plot_training_curve, plot_training_curve_by_family
FIG_DIR = os.path.join(REPO_DIR, 'docs', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
plot_training_curve(
    overall, os.path.join(FIG_DIR, 'training_curve.png'),
    baseline_scores={'heuristic': heur_overall, 'naive': 0.0},
)
plot_training_curve_by_family(
    grouped, os.path.join(FIG_DIR, 'training_curve_by_family.png'),
    family_order=['easy', 'medium', 'hard', 'nightmare'],
)
print(f'figures saved to {FIG_DIR}/')

figures saved to /content/chargebackops/docs/figures/


## 5. Diagnose final checkpoint

Print the trained checkpoint's completion vs. the heuristic oracle on three representative tasks (easy, hard, nightmare). Verifies the model emits valid JSON and shows what the trained policy actually does.

In [ ]:
from training.env_adapter import build_prompt, parse_completion
from training.outcome_reward import compute_outcome_reward
from server.chargeback_ops_environment import ChargebackOpsEnvironment
from runners.benchmark_runner import heuristic_policy

print("diagnose adapter (active GRPO model from rescue cell)")


def diag_policy(prompt):
    chat = eval_tok.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = eval_tok(
        chat,
        return_tensors='pt',
        truncation=True,
        max_length=1024
    ).to(m.device)

    with torch.no_grad():
        out = m.generate(
            **inputs,
            max_new_tokens=192,
            do_sample=False,
            pad_token_id=eval_tok.eos_token_id,
            eos_token_id=eval_tok.eos_token_id,
        )

    return eval_tok.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )


for tid in ['goods_not_received_easy', 'queue_optimization_hard', 'generated_nightmare_s31']:
    env = ChargebackOpsEnvironment()
    obs = env.reset(task_id=tid)

    raw = build_prompt(obs.model_dump())
    completion = diag_policy(raw)
    parsed = parse_completion(completion)

    oracle = heuristic_policy(obs.model_dump())
    pnl = compute_outcome_reward(
        ['x'],
        [completion],
        task_ids=[tid],
        state_steps=[0]
    )[0]

    print(f"\n=== {tid} ===")
    print(f"oracle: {oracle.action_type} case={oracle.case_id}")
    print(f"completion (first 200): {repr(completion[:200])}")
    print(f"parsed: {parsed}")
    print(f"outcome PnL (normalized): {pnl:+.3f}")

diagnose adapter (active GRPO model from rescue cell)

=== goods_not_received_easy ===
oracle: select_case case=CB-E1
completion (first 200): '{"action_type":"select_case","case_id":"CB-E1","metadata":{}}'
parsed: {'action_type': 'select_case', 'case_id': 'CB-E1'}
outcome PnL (normalized): +1.000

=== queue_optimization_hard ===
oracle: select_case case=CB-H3
completion (first 200): '{"action_type":"select_case","case_id":"CB-H3","metadata":{}}'
parsed: {'action_type': 'select_case', 'case_id': 'CB-H3'}
outcome PnL (normalized): +0.211

=== generated_nightmare_s31 ===
oracle: select_case case=CB-G3
completion (first 200): '{"action_type":"select_case","case_id":"CB-G5","metadata":{}}'
parsed: {'action_type': 'select_case', 'case_id': 'CB-G5'}
outcome PnL (normalized): -0.636


## Done

Artifacts written to `docs/figures/`:
* `training_curve.png` - overall mean rubric score across SFT + GRPO checkpoints, with heuristic baseline.
* `training_curve_by_family.png` - per-difficulty curves (easy / medium / hard / nightmare).

Adapter weights (under `PERSIST_ROOT`):
* `sft-merchant-agent/final/` - Phase A output.
* `grpo-merchant-agent/final/` - Phase B output (GRPO with outcome reward).

To use the trained model:
```python
from peft import PeftModel
sft_model = PeftModel.from_pretrained(base, 'sft-merchant-agent/final')
merged = sft_model.merge_and_unload()
trained = PeftModel.from_pretrained(merged, 'grpo-merchant-agent/final')
```